# 二吸収帯整合型 SC-LMMF のための合成メタンプルーム注入・評価

HISUIの実観測背景スペクトルに既知濃度の合成メタンプルームを注入し、1.6 µm帯と2.3 µm帯の両方を用いる推定手法を評価する。

## 評価の考え方

合成プルーム注入前の観測cubeを $L_{\mathrm{bg}}$、MODTRAN LUTから求めた濃度増分 $\alpha$ に対する放射輝度比を $T(\lambda,\alpha)$ とする。

合成プルームを注入したスペクトルは、次式で生成する。

$$
L_{\mathrm{syn}}(x,y,\lambda)
=
L_{\mathrm{bg}}(x,y,\lambda)
T\left(
\lambda,
\alpha_{\mathrm{true}}(x,y)
\right)
$$

ここで、

- $L_{\mathrm{syn}}(x,y,\lambda)$：合成プルーム注入後の放射輝度
- $L_{\mathrm{bg}}(x,y,\lambda)$：注入前の背景放射輝度
- $\alpha_{\mathrm{true}}(x,y)$：注入するメタン濃度増分の真値
- $T(\lambda,\alpha)$：MODTRAN LUTから求めた放射輝度比

である。

放射輝度比は、0 ppm増分のMODTRANスペクトルを基準として次式で求める。

$$
T(\lambda,\alpha)
=
\frac{
L_{\mathrm{MODTRAN}}(\lambda,\alpha)
}{
L_{\mathrm{MODTRAN}}(\lambda,0)
}
$$

この比を用いる場合、MODTRAN放射輝度を100倍してHISUIと単位を合わせても、分子と分母の両方に同じ係数が掛かるため、その係数は相殺される。

したがって、合成プルーム注入の処理では、MODTRAN放射輝度を100倍する必要はない。

`CH4b.csv` の各列名が背景濃度からのメタン濃度増分をppm単位で表している場合、$\alpha_{\mathrm{true}}$ と推定値 $\hat{\alpha}$ の単位もppmとなる。

## 重要な評価段階

### Stage A：Oracle background評価

注入前の観測cubeを真の背景スペクトルとして使用する。

この評価では、背景推定誤差を含めずに、次の要素を確認する。

- 合成プルーム注入処理
- MODTRAN LUT
- UAS
- 1.6 µm帯と2.3 µm帯の推定精度
- 二吸収帯融合
- 非線形フィッティング

### Stage B：Blind background評価

注入前の真の背景スペクトルを直接使用せず、Iterative MFやSVD低ランク再構成などによって背景スペクトルを推定する。

この評価では、実運用に近い条件で次の性能を確認する。

- 背景推定誤差の影響
- 地表面スペクトル変動への頑健性
- プルーム画素による背景統計量の汚染
- 偽陽性の発生
- 濃度推定精度

まずStage Aにおいて、既知の $\alpha_{\mathrm{true}}$ が正しく回収できることを確認する。

その後、真の背景スペクトルを既存のIterative MFやSVD背景再構成による推定背景へ置き換え、Stage Bの評価を行う。

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import least_squares

np.set_printoptions(precision=5, suppress=True)
